In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
from pathlib import Path

In [2]:
adata_cd8 = sc.read_h5ad("/scratch/user/s4575250/BIOX7014_Thesis/write/03_batch_expression/PICA_Batch001-Batch007/PICA_Batch001-Batch007_cd8_combined_annot_with_age_scvi.h5ad")

In [3]:
adata_cd8

AnnData object with n_obs × n_vars = 265568 × 38606
    obs: 'status', 'assignment', 'pica_id', 'pool_id', 'sequencing_batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'status_manual', 'cell_type', 'broad_cell_type', 'pica_broad_cell_type', 'pica_cell_type', 'pica_cell_type_01_myeloid', 'pica_cell_type_02_b', 'pica_cell_type_03_nk_t', 'pica_cell_type_complete', 'pica_broad_cell_type_complete', 'pica_broad_cell_type_complete_01_gdT_corrected', 'pica_cell_type_complete_01_gdT_corrected', 'status_manual_01_pica', 'Record ID', 'Sex', 'Age_years', 'Age', 'Event Name', 'Was baseline blood sample collected?', 'IFC ID', "Child's sex", 'Age in 

In [5]:
# set metadata column
donor_col = "pica_id"
celltype_col = "pica_cell_type_complete_01_gdT_corrected"
age_col = "Age_years"
sex_col = "Sex"
batch_col = "sequencing_batch"

In [6]:
print(adata_cd8.shape)
print(adata_cd8.obs[celltype_col].value_counts())

(265568, 38606)
pica_cell_type_complete_01_gdT_corrected
CD8 naive/Tcm          175694
CD8 Tem                 50185
CD8 GZMK+ naive/Tcm     39689
Name: count, dtype: int64


In [10]:
# Subset metadata and layers only
obs = adata_cd8.obs.copy()

cd8_mask = obs[celltype_col].astype(str).isin([
    "CD8 naive/Tcm",
    "CD8 Tem",
    "CD8 GZMK+ naive/Tcm"
])

required_cols = [donor_col, celltype_col, age_col, sex_col, batch_col]
valid_mask = obs[required_cols].notna().all(axis=1)

keep_mask = cd8_mask & valid_mask

obs_cd8 = obs.loc[keep_mask].copy()

print(obs_cd8.shape)
print(obs_cd8[celltype_col].value_counts())

(265568, 76)
pica_cell_type_complete_01_gdT_corrected
CD8 naive/Tcm          175694
CD8 Tem                 50185
CD8 GZMK+ naive/Tcm     39689
Name: count, dtype: int64


In [12]:
# extract raw counts
X = adata_cd8.layers["counts"]

if sparse.issparse(X):
    X = X.tocsr()

genes = adata_cd8.var_names.astype(str)

# Get original integer positions of kept CD4 cells
cell_positions = np.where(keep_mask.values)[0]

print(X.shape)

(265568, 38606)


## Create Pseudobulk

In [13]:
# donor × cell type
obs_cd8["pseudobulk_id"] = (
    obs_cd8[donor_col].astype(str) + "__" + obs_cd8[celltype_col].astype(str)
)

# number of pseudobulk samples
print(obs_cd8["pseudobulk_id"].nunique())
print(obs_cd8["pseudobulk_id"].value_counts().describe())

383
count     383.000000
mean      693.389034
std       625.704715
min         1.000000
25%       233.000000
50%       410.000000
75%      1116.000000
max      2971.000000
Name: count, dtype: float64


In [14]:
# Aggregate raw counts
pb_counts = []
pb_meta = []

for pb_id, group_index in obs_cd8.groupby("pseudobulk_id").indices.items():
    
    # group_index = positions within obs_cd8
    # original_positions = positions within original adata/X
    original_positions = cell_positions[group_index]
    meta_sub = obs_cd8.iloc[group_index]
    
    summed_counts = np.asarray(X[original_positions, :].sum(axis=0)).ravel()
    
    pb_counts.append(summed_counts)
    
    pb_meta.append({
        "sample_id": pb_id,
        "donor_id": meta_sub[donor_col].iloc[0],
        "cell_type": meta_sub[celltype_col].iloc[0],
        "age": meta_sub[age_col].iloc[0],
        "sex": meta_sub[sex_col].iloc[0],
        "batch": meta_sub[batch_col].iloc[0],
        "n_cells": len(original_positions)
    })

pb_counts_df = pd.DataFrame(
    pb_counts,
    index=[m["sample_id"] for m in pb_meta],
    columns=genes
)

pb_meta_df = pd.DataFrame(pb_meta).set_index("sample_id")

print("Pseudobulk count matrix:")
print(pb_counts_df.shape)

print("\nPseudobulk metadata:")
print(pb_meta_df.shape)

print("\nPseudobulk samples per cell type:")
print(pb_meta_df["cell_type"].value_counts())

Pseudobulk count matrix:
(383, 38606)

Pseudobulk metadata:
(383, 6)

Pseudobulk samples per cell type:
cell_type
CD8 GZMK+ naive/Tcm    128
CD8 naive/Tcm          128
CD8 Tem                127
Name: count, dtype: int64


In [15]:
all_donors = set(pb_meta_df["donor_id"].unique())

cytot_donors = set(
    pb_meta_df.loc[pb_meta_df["cell_type"] == "CD8 Tem", "donor_id"]
)

missing_cytot_donors = all_donors - cytot_donors

missing_cytot_donors

{'PICA0006'}

In [16]:
obs_cd8.loc[
    obs_cd8["pica_id"].isin(missing_cytot_donors),
    celltype_col
].value_counts()

pica_cell_type_complete_01_gdT_corrected
CD8 naive/Tcm          636
CD8 GZMK+ naive/Tcm     27
CD8 Tem                  0
Name: count, dtype: int64

In [17]:
# filter low-cell pseudobulk samples
min_cells = 20

keep_samples = pb_meta_df["n_cells"] >= min_cells

pb_counts_df = pb_counts_df.loc[keep_samples]
pb_meta_df = pb_meta_df.loc[keep_samples]

print("After filtering pseudobulk samples with n_cells <", min_cells)
print("Counts:", pb_counts_df.shape)
print("Metadata:", pb_meta_df.shape)

print("\nSamples per cell type after filtering:")
print(pb_meta_df["cell_type"].value_counts())

print("\nCell counts per cell type after filtering:")
print(pb_meta_df.groupby("cell_type")["n_cells"].sum())

After filtering pseudobulk samples with n_cells < 20
Counts: (377, 38606)
Metadata: (377, 6)

Samples per cell type after filtering:
cell_type
CD8 GZMK+ naive/Tcm    128
CD8 naive/Tcm          128
CD8 Tem                121
Name: count, dtype: int64

Cell counts per cell type after filtering:
cell_type
CD8 GZMK+ naive/Tcm     39689
CD8 Tem                 50114
CD8 naive/Tcm          175694
Name: n_cells, dtype: int64


In [18]:
pb_counts_df.T.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD8_pseudobulk_counts_gene_by_sample.csv")
pb_meta_df.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD8_pseudobulk_metadata.csv")

In [19]:
# Sanity checks
counts_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD8_pseudobulk_counts_gene_by_sample.csv", index_col=0)
meta_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD8_pseudobulk_metadata.csv", index_col=0)

print("Counts check:", counts_check.shape)
print("Metadata check:", meta_check.shape)

assert list(counts_check.columns) == list(meta_check.index)

print("Sample names match between counts and metadata.")
print(meta_check.head())

Counts check: (38606, 377)
Metadata check: (377, 6)
Sample names match between counts and metadata.
                               donor_id            cell_type    age   sex  \
sample_id                                                                   
PICA0001__CD8 GZMK+ naive/Tcm  PICA0001  CD8 GZMK+ naive/Tcm   2.09  Male   
PICA0001__CD8 Tem              PICA0001              CD8 Tem   2.09  Male   
PICA0001__CD8 naive/Tcm        PICA0001        CD8 naive/Tcm   2.09  Male   
PICA0002__CD8 GZMK+ naive/Tcm  PICA0002  CD8 GZMK+ naive/Tcm  15.09  Male   
PICA0002__CD8 naive/Tcm        PICA0002        CD8 naive/Tcm  15.09  Male   

                                                                    batch  \
sample_id                                                                   
PICA0001__CD8 GZMK+ naive/Tcm  20240530_WGS_20240530_sc_PICA0001-PICA0007   
PICA0001__CD8 Tem              20240530_WGS_20240530_sc_PICA0001-PICA0007   
PICA0001__CD8 naive/Tcm        20240530_WGS_20240530